# Explanation of the beginning of the article

*Radford M. Neal (2000), « Markov Chain Sampling Methods for Dirichlet Process Mixture Models »*


## 1. Introduction

Suppose you observe data $y_1, \dots, y_n$. A mixture model assumes the data comes from several different distribution (for example a mixture of Gaussian). Meaning that each observation is linked to a source and each of the sources have different distribution.

The main issue with this approach is that usualy one needs to pick the number of components $K$ (or sources) in advance. This task comes with its own set of difficulties, that is why a different approach is to have infinitely many components and have a prior keeping only a few of them with most of the weight. The Dirichlet process is such a prior. The output is a posterior distribution over the number of components.

However this approach has some issues. The posterior cannot be computed exactly beyond a small number of observations (for 9 observations there are 21,147 partitions possible). That is why Markov chain Monte Carlo (MCMC) are used.

---

## 2. Description of the model

The data $y_1, \dots, y_n$ are treated as part of an indefinite exchangeable sequence, equivalently as independent draws from some unknown distribution. Each $y_i$ may be multivariate, with real-valued or categorical components. The model is as follows

$$
\begin{aligned}
y_i \mid \theta_i &\sim F(\theta_i) \\
\theta_i \mid G &\sim G \\
G &\sim \mathrm{DP}(G_0, \alpha)
\end{aligned}
\tag{2.1}
$$

Meaning that observation $i$ comes from a distribution in the family $F$, with parameter $\theta_i$. Each observation's parameter is drawn from a random distribution $G$ (named the mixing distribution). Finally $G$ itself is random, with a Dirichlet process prior.(a Dirichlet process is a distribution over distributions).

The Dirichlet process has two parameters. First the Base distribution $G_0$. This defines what a parameter looks like. The base measure and the mean of the DP are $\alpha G_0$ and $G_0$. Then the concentration parameter $\alpha > 0$. This parameter controls how $G$ deviates from $G_0$ meaning that small $\alpha$ make it so that the different parameters taken from these distribution are close to each other so there are not a lot of different distributions in the mixture. Inversely a big $\alpha$ means that the parameters are far from each other so a big number of distributions in the mixture is more likely.

---

## 3. First approach

A first approach for $\theta_i$ is :

$$
\theta_i \mid \theta_1, \dots, \theta_{i-1}
\;\sim\;
\frac{1}{i-1+\alpha} \sum_{j=1}^{i-1} \delta(\theta_j)
\;+\;
\frac{\alpha}{i-1+\alpha}\, G_0
\tag{2.2}
$$

This means that observation $i$ has two possibilities given the $i-1$ previous ones:

- Either it copies a previous value. With total probability $\frac{i-1}{i-1+\alpha}$, it picks one of the previous observations uniformly and copies its $\theta_j$ (a cluster is chosen with probability proportional to its size).
- Either it starts a new cluster. With probability $\frac{\alpha}{i-1+\alpha}$, meaning it draws a fresh value from $G_0$.

An example with $\alpha = 1$ :

- **Observation 1:** probability of a new cluster is $\alpha/(0+\alpha) = 1$. It always starts the first cluster.
- **Observation 2:** same cluster as observation 1 with probability $1/(1+1) = 1/2$. New cluster with probability $1/2$.
- **Observation 3, if 1 and 2 are together:** joins their cluster with probability $2/3$, starts a new one with probability $1/3$.
- **Observation 3, if 1 and 2 are apart:** joins cluster 1 with probability $1/3$, joins cluster 2 with probability $1/3$, starts a new one with probability $1/3$.

The joint distribution of $(\theta_1, \dots, \theta_n)$ does not depend on the order of the observations.

---

## 4. Second approach

This second formulation is the one the algorithms are derived from.

$$
\begin{aligned}
y_i \mid c_i, \boldsymbol\phi &\sim F(\phi_{c_i}) \\
c_i \mid \mathbf p &\sim \mathrm{Discrete}(p_1, \dots, p_K) \\
\phi_c &\sim G_0 \\
\mathbf p &\sim \mathrm{Dirichlet}(\alpha/K, \dots, \alpha/K)
\end{aligned}
\tag{2.3}
$$

$c_i$ is the class of observation $i$, meaning it says which component generated $y_i$. $\phi_c$ are the parameters of the class $c$. $\mathbf p$ is the weight of proportions. The link between the two approaches is that $\theta_i = \phi_{c_i}$.

Using the formula on conditionnal probability it is possible to write the probability that observation $i$ is in cluster $c$ as : 

$$
P(c_i = c \mid c_1, \dots, c_{i-1})
= \frac{P(c_1, \dots, c_{i-1}, c_i = c)}{P(c_1, \dots, c_{i-1})}
\tag{2.4}
$$

Then we can write each joint probability as an integral over $\mathbf p$ against the Dirichlet density:

$$
= \frac{\int p_{c_1}\cdots p_{c_{i-1}}\, p_c \;\;\Gamma(\alpha)\,\Gamma(\alpha/K)^{-K}\, p_1^{\alpha/K-1}\cdots p_K^{\alpha/K-1}\, d\mathbf p}
       {\int p_{c_1}\cdots p_{c_{i-1}} \;\;\Gamma(\alpha)\,\Gamma(\alpha/K)^{-K}\, p_1^{\alpha/K-1}\cdots p_K^{\alpha/K-1}\, d\mathbf p}
\tag{2.5}
$$

Using the identity of the Dirichlet distribution these integrals can be further rewritten as :

$$
P(c_i = c \mid c_1, \dots, c_{i-1}) = \frac{n_{i,c} + \alpha/K}{i - 1 + \alpha}
\tag{2.6}
$$

where $n_{i,c}$ is the number of $c_j$ with $j < i$ that equal $c$.

Then by taking the limit we have :

$$
P(c_i = c \mid c_1,\dots,c_{i-1}) \;\to\; \frac{n_{i,c}}{i-1+\alpha}
\qquad\text{(class } c \text{ already used)}
$$

$$
P(c_i \ne c_j \text{ for all } j<i \mid c_1,\dots,c_{i-1}) \;\to\; \frac{\alpha}{i-1+\alpha}
\qquad\text{(a new class)}
\tag{2.7}
$$

The second line arises because there are $K - k$ unused classes (with $k$ the number in use), each with probability $\frac{\alpha/K}{i-1+\alpha}$, and $(K-k)\frac{\alpha/K}{i-1+\alpha} \to \frac{\alpha}{i-1+\alpha}$.

We obtain exactly the same formulation !

---

## 5. Why the second approach matters

Algorithms for the infinite model come from the finite model

- Write the sampler for finite $K$ (where all quantities are explicit and finite).
- Let $K \to \infty$. Classes that no observation uses are never represented explicitly, so the state only contains the classes currently occupied.
- Conversely, every algorithm for Dirichlet process mixtures has a counterpart for finite mixtures. 


---

## 6. Rest of the paper

1. **Section 3** uses these two views to write Gibbs samplers when $G_0$ is conjugate to $F$ (Algorithms 1 to 3).
2. **Section 4** explains why those samplers fail for non-conjugate $G_0$, and reviews existing fixes (including Algorithm 4, "no gaps").
3. **Section 5** gives the Metropolis–Hastings samplers (Algorithms 5 to 7), proposing from the conditional prior, which is exactly (2.7) / (5.4).
4. **Section 6** gives the auxiliary-parameter Gibbs sampler (Algorithm 8).
5. **Section 7** adds hyperparameters, and **Section 8** tests Algorithms 4 to 8 on a 9-point dataset.

**The central idea to carry forward:** the prior for the cluster structure is simple and explicit, as in (2.7) or (3.1). The only real difficulty in posterior sampling is how to account for the **likelihood** of a possible *new* cluster, because that requires integrating $F(y_i,\phi)$ over $G_0$. All the algorithms differ in how they handle that one integral.